# SAP Maroc — Benchmarking de 4 modèles de prévision météo 7 jours

**Thèse** : *Conception et Implémentation d'un Système d'Alerte Précoce Intelligent pour les Vagues de Chaleur et de Froid au Maroc*

---

## Modèles comparés

| Modèle | Type | Description |
|---|---|---|
| 1. **ConvLSTM** | Spatiotemporel récurrent | Seq2Seq encodeur-décodeur à 2 couches |
| 2. **3D‑CNN** | Spatiotemporel non‑récurrent | Convolutions 3D + pooling adaptatif |
| 3. **PredRNN** | Spatiotemporel récurrent | ST‑LSTM à deux couches |
| 4. **Persistence** | Baseline naïve | Dernière observation répétée 7 fois |

**Input** : 7 jours × 11 canaux (`tmax, tmin, rh, u10_mean, v10_mean, z500, t850, ws10, ssrd_mean, sp_mean, tp_sum`)  
**Output** : 7 jours × 3 cibles (`tmax, tmin, rh`)  
**Données** : ERA5-Land 1990–2025, 0.25° résolution, grille Maroc 37×65

## 0. Configuration & Montage Google Drive

In [2]:
!pip install xarray
# !pip install h5netcdf
# !pip install netCDF4

In [3]:
# ── Montage Google Drive ──
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, csv, logging
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

import torch
import torch.nn as nn
import torch.nn.functional as F

logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s | %(levelname)-8s | %(message)s')
log = logging.getLogger('Benchmark')

print(f'PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision('high')
print(f'Device: {DEVICE}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PyTorch 2.11.0+cu128  |  CUDA: True
Device: cuda


In [4]:
# ── Configuration ──
@dataclass
class BenchmarkConfig:
    data_path:       str = '/content/drive/MyDrive/ERA5_Morocco_ConvLSTM_1/era5_full_grid_1980_2025_13features.npz'
    output_dir:      str = '/content/drive/MyDrive/training_artifacts_npz'
    input_window:    int = 7
    output_window:   int = 7
    train_ratio:    float = 0.70
    val_ratio:      float = 0.15
    test_ratio:     float = 0.15
    batch_size:      int = 24          # OPTIMISATION: augmenté de 8 à 24 pour accélérer (réduit les goulots d'étranglement)
    epochs:          int = 100
    learning_rate:  float = 5e-4
    filters:         int = 64          # ConvLSTM
    predrnn_hidden:  int = 64          # FIX: 64 (était 32) → comparaison équitable
    tf_ratio_start: float = 0.5        # NEW: scheduled sampling (teacher forcing)
    use_amp:        bool = True        # OPTIMISATION: activé (Précision mixte 16-bit) pour x2 vitesse
    seed:            int = 42
    # 13 features from the .npz array (in the exact order they were concatenated)
    channels:        Sequence[str] = (
        'Tmax', 'Tmin', 'u_wind', 'v_wind', 'Tdew', 'SoilT',
        'Solar', 'Press', 'SoilM', 'WindSpeed', 'RH'
    )

    # The targets we want to predict
    targets:         Sequence[str] = ('HeatIndex', 'WindChill')

    def __post_init__(self):
        self.n_in = len(self.channels)
        self.n_out = len(self.targets)

cfg = BenchmarkConfig()
OUTPUT_DIR = Path(cfg.output_dir)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.seed)

print(f'Input:  {cfg.input_window}j × {cfg.n_in} canaux')
print(f'Output: {cfg.output_window}j × {cfg.n_out} cibles {cfg.targets}')
print(f'Batch:  {cfg.batch_size}  |  Epochs: {cfg.epochs}  |  Filters: {cfg.filters}')
print(f'AMP (Précision Mixte): {cfg.use_amp}')


Input:  7j × 11 canaux
Output: 7j × 2 cibles ('HeatIndex', 'WindChill')
Batch:  24  |  Epochs: 100  |  Filters: 64
AMP (Précision Mixte): True


In [5]:
import numpy as np

# Path to the final master dataset
final_file = '/content/drive/MyDrive/ERA5_Morocco_ConvLSTM_1/era5_full_grid_1980_2025_13features.npz'

try:
    print(f"Opening {final_file}...")
    data = np.load(final_file)

    print("\n--- NPZ Contents ---")
    print(f"Keys found: {data.files}")

    for key in data.files:
        arr = data[key]
        print(f"{key}: shape {arr.shape}, dtype {arr.dtype}")

except Exception as e:
    print(f"Error loading or checking file: {e}")

Opening /content/drive/MyDrive/ERA5_Morocco_ConvLSTM_1/era5_full_grid_1980_2025_13features.npz...

--- NPZ Contents ---
Keys found: ['data', 'valid_mask', 'height', 'width']
data: shape (16756, 153, 163, 13), dtype float32
valid_mask: shape (153, 163), dtype bool
height: shape (), dtype int64
width: shape (), dtype int64


## 1. Chargement & Exploration des données

In [ ]:
# ── Ouverture du fichier NPZ ──
npz_file = np.load(cfg.data_path)
print('Clés disponibles dans le fichier NPZ :', npz_file.files)

# Extraction du tenseur principal
tensor_raw = npz_file['data']
# print('Dimensions du tenseur de données :', tensor_raw.shape)

# if len(tensor_raw.shape) == 4:
#     # Assuming shape is (T, H, W, C) or similar
#     print(f'Nombre de canaux dans les données: {tensor_raw.shape[-1]}')
#     if tensor_raw.shape[-1] == len(cfg.channels):
#         print('✅ Le nombre de canaux correspond à la configuration.')
#     else:
#         print(f'⚠️ Incohérence : attendu {len(cfg.channels)} canaux, trouvé {tensor_raw.shape[-1]}')
# else:
#     print('⚠️ Format inattendu du tenseur.')


Clés disponibles dans le fichier NPZ : ['data', 'valid_mask', 'height', 'width']


In [ ]:
# # ── Statistiques rapides par canal ──
# print("Calcul des statistiques sur le tenseur principal...")
# for i, var in enumerate(cfg.channels):
#     if i < tensor_raw.shape[-1]:
#         v = tensor_raw[..., i]
#         print(f'{var:>10}: min={np.nanmin(v):8.2f}  max={np.nanmax(v):8.2f}  mean={np.nanmean(v):8.2f}  std={np.nanstd(v):8.2f}')


## 2. Prétraitement — Normalisation & Fenêtres glissantes

In [ ]:
def make_windows(tensor, cfg):
    """Convert (T, H, W, C) → (N, window, H, W, C) pour X et Y."""
    WIN = cfg.input_window + cfg.output_window
    T = tensor.shape[0]
    X_list, Y_list = [], []
    for i in range(0, T - WIN + 1):
        X_list.append(tensor[i : i + cfg.input_window])
        Y_list.append(tensor[i + cfg.input_window : i + WIN])
    if not X_list:
        raise ValueError(f'Tensor trop court ({T}j) pour fenêtre {WIN}j.')
    return np.stack(X_list, axis=0), np.stack(Y_list, axis=0)

In [ ]:
# ── Nettoyage et préparation du tenseur ──
# Le tenseur est déjà sous la forme (T, H, W, C) dans le fichier NPZ
tensor = tensor_raw.astype(np.float32)

# Remplacer les éventuels NaN par 0.0
# np.nan_to_num(tensor, copy=False, nan=0.0)

T_total, H, W, C = tensor.shape

# print(f'Tenseur brut : {tensor.shape}  (T={T_total}, H={H}, W={W}, C={cfg.n_in})')
# print(f'Mémoire RAM : {tensor.nbytes / 1e9:.2f} Go')

In [ ]:
import pandas as pd

# ── Split chronologique 70/15/15 avec buffer ──
n_test = max(1, int(T_total * cfg.test_ratio))
n_val  = max(1, int(T_total * cfg.val_ratio))
buffer = cfg.output_window * 2

train_end = T_total - n_val - n_test - buffer * 2
if train_end <= 0:
    raise ValueError('Dataset trop petit.')

raw_train = tensor[:train_end]
raw_val   = tensor[train_end + buffer : train_end + buffer + n_val]
raw_test  = tensor[train_end + buffer + n_val + buffer :]

# NEW: dates alignées sur chaque split (pour climatologie & seuils percentiles)
# Génération des dates puisque le NPZ ne contient pas la dimension time (début: 1980-01-01)
times = pd.date_range(start='1980-01-01', periods=T_total, freq='D').values
times_train = times[:train_end]
times_val   = times[train_end + buffer : train_end + buffer + n_val]
times_test  = times[train_end + buffer + n_val + buffer :]

print(f'Train : {raw_train.shape[0]} jours  ({str(times_train[0])[:10]} → {str(times_train[-1])[:10]})')
print(f'Val   : {raw_val.shape[0]} jours  ({str(times_val[0])[:10]} → {str(times_val[-1])[:10]})')
print(f'Test  : {raw_test.shape[0]} jours  ({str(times_test[0])[:10]} → {str(times_test[-1])[:10]})')

# Résolution de la grille
dlat = 0.1
dlon = 0.1
print(f'Résolution grille : {dlat:.3f}° lat × {dlon:.3f}° lon')


In [ ]:
from numpy.lib.stride_tricks import sliding_window_view

# ── Z-score (Standardization) In-Place pour économiser la RAM ──
print("Calcul des Moyennes/Ecart-types sur le jeu d'entraînement...")
c_means = np.zeros(cfg.n_in, dtype=np.float32)
c_stds  = np.zeros(cfg.n_in, dtype=np.float32)

for c in range(cfg.n_in):
    c_means[c] = raw_train[..., c].mean()
    c_stds[c]  = raw_train[..., c].std()

# Eviter la division par zéro
c_stds[c_stds < 1e-6] = 1.0

print("Application de la normalisation Z-score...")
for c in range(cfg.n_in):
    raw_train[..., c] = (raw_train[..., c] - c_means[c]) / c_stds[c]
    raw_val[..., c]   = (raw_val[..., c]   - c_means[c]) / c_stds[c]
    raw_test[..., c]  = (raw_test[..., c]  - c_means[c]) / c_stds[c]

# ── Création des fenêtres sans dupliquer la mémoire ──
def make_windows_view(tensor, cfg):
    WIN = cfg.input_window + cfg.output_window
    # (T, H, W, C) -> (T-WIN+1, H, W, C, WIN)
    view = sliding_window_view(tensor, window_shape=WIN, axis=0)
    # Déplacer l'axe temporel WIN à la position 1 -> (T-WIN+1, WIN, H, W, C)
    view = np.moveaxis(view, -1, 1)

    X = view[:, :cfg.input_window, ...]
    Y = view[:, cfg.input_window:, ...]
    return X, Y

print("Création des fenêtres glissantes (vues mémoire)...")
X_train, Y_train_raw = make_windows_view(raw_train, cfg)
X_val,   Y_val_raw   = make_windows_view(raw_val, cfg)
X_test,  Y_test_raw  = make_windows_view(raw_test, cfg)

# Ne garder que les canaux configurés pour X, et les cibles pour Y
X_train = X_train[..., :cfg.n_in]
X_val   = X_val[..., :cfg.n_in]
X_test  = X_test[..., :cfg.n_in]

Y_train = Y_train_raw[..., :cfg.n_out]
Y_val   = Y_val_raw[  ..., :cfg.n_out]
Y_test  = Y_test_raw[ ..., :cfg.n_out]

norm_stats = {
    'mean': c_means.astype(np.float32),
    'std':  c_stds.astype(np.float32),
}

print(f'X_train : {X_train.shape}')
print(f'Y_train : {Y_train.shape}\n')
print(f'Mean par canal : {norm_stats["mean"]}')
print(f'Std par canal  : {norm_stats["std"]}')


## Correlation heatmap

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ── Correlation heatmap (inputs) — sur les PIXELS VALIDES d'entraînement ──
print("Chargement du masque des pixels valides (hors océan/NaN)...")
valid_mask = np.load(cfg.data_path)['valid_mask']

print("Extraction des données sur la zone valide...")
# Extraction uniquement des pixels terrestres valides pour les canaux configurés : (T, H, W, C) -> (T, N_valid, cfg.n_in)
valid_data = raw_train[:, valid_mask, :cfg.n_in]
# Aplatir la dimension spatio-temporelle : -> (T * N_valid, C)
valid_data = valid_data.reshape(-1, cfg.n_in)

C = cfg.n_in
print("Calcul de la matrice de corrélation de Pearson (Low-RAM float64)...")
corr = np.zeros((C, C), dtype=np.float64)

# 1. Moyennes et Std calculées exactement en float64 pour éviter l'explosion numérique
v_mean = np.zeros(C, dtype=np.float64)
v_std  = np.zeros(C, dtype=np.float64)

for i in range(C):
    col = valid_data[:, i].astype(np.float64)
    v_mean[i] = np.mean(col)
    v_std[i]  = np.std(col)
    if v_std[i] < 1e-6:
        v_std[i] = 1.0

# 2. Corrélation calculée 2 colonnes à la fois (prend ~2 Go de RAM au lieu de 12+ Go)
for i in range(C):
    xi = (valid_data[:, i].astype(np.float64) - v_mean[i]) / v_std[i]
    corr[i, i] = 1.0
    for j in range(i + 1, C):
        xj = (valid_data[:, j].astype(np.float64) - v_mean[j]) / v_std[j]
        c_val = np.mean(xi * xj)
        corr[i, j] = c_val
        corr[j, i] = c_val

# Libérer la mémoire des grands tenseurs
del valid_data, xi, xj, col

# ── Affichage ──
fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap='coolwarm')
ax.set_xticks(range(C))
ax.set_yticks(range(C))
ax.set_xticklabels(cfg.channels, rotation=45, ha='right', fontsize=10)
ax.set_yticklabels(cfg.channels, fontsize=10)
ax.set_title('Correlation Heatmap (Valid Land Pixels Only)', fontsize=14, fontweight='bold')

# Annoter les valeurs de correlation
for i in range(C):
    for j in range(C):
        val = corr[i, j]
        color = 'white' if abs(val) > 0.6 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=8, color=color)

fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()

corr_path = OUTPUT_DIR / 'correlation_heatmap_input_valid.png'
fig.savefig(corr_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {corr_path}')


## 3. Définition des modèles

### 3a. ConvLSTM Seq2Seq (modèle principal)

In [ ]:
class ConvLSTMCell(nn.Module):
    def __init__(self, in_ch, hid_ch, ks=3):
        super().__init__()
        self.conv = nn.Conv2d(in_ch + hid_ch, 4 * hid_ch, ks, padding=ks // 2)

    def forward(self, x, h, c):
        combined = torch.cat([x, h], dim=1)
        gates = self.conv(combined)
        i, f, g, o = torch.chunk(gates, 4, dim=1)
        c_next = torch.sigmoid(f) * c + torch.sigmoid(i) * torch.tanh(g)
        h_next = torch.sigmoid(o) * torch.tanh(c_next)
        return h_next, c_next


class Seq2SeqConvLSTM(nn.Module):
    """Encodeur 2 couches → Décodeur 1 couche auto-régressif.
    FIX: n_out paramétrique (plus de '3' codé en dur) + scheduled sampling."""
    def __init__(self, cfg):
        super().__init__()
        f = cfg.filters
        self.n_out   = cfg.n_out
        self.out_win = cfg.output_window
        self.enc1 = ConvLSTMCell(cfg.n_in, f)
        self.enc2 = ConvLSTMCell(f, f)
        self.dec  = ConvLSTMCell(cfg.n_out, f)
        self.head = nn.Conv2d(f, cfg.n_out, 3, padding=1)

    def forward(self, x, H=None, W=None, y=None, tfr=0.0):
        B, T, C, H_in, W_in = x.size()
        f = self.head.in_channels
        h1 = c1 = torch.zeros(B, f, H_in, W_in, device=x.device)
        h2 = c2 = torch.zeros(B, f, H_in, W_in, device=x.device)

        for t in range(T):
            h1, c1 = self.enc1(x[:, t], h1, c1)
            h2, c2 = self.enc2(h1, h2, c2)

        h_dec, c_dec = h2, c2
        dec_in = torch.zeros(B, self.n_out, H_in, W_in, device=x.device)
        outputs = []
        for t in range(self.out_win):
            h_dec, c_dec = self.dec(dec_in, h_dec, c_dec)
            out_t = self.head(h_dec)
            outputs.append(out_t.unsqueeze(1))
            # Scheduled sampling : pendant l'entraînement, on injecte parfois
            # la vérité terrain pour stabiliser l'apprentissage long-horizon.
            if (y is not None) and (torch.rand(1).item() < tfr):
                dec_in = y[:, t]
            else:
                dec_in = out_t
        return torch.cat(outputs, dim=1)


### 3b. 3D‑CNN (baseline non‑récurrent)

In [ ]:
class CNN3D(nn.Module):
    """3D ConvNet — FIX MAJEUR : l'ancienne version faisait
    adaptive_avg_pool3d(x, (1,H,W)) qui ÉCRASAIT toute la dimension temporelle
    en une carte statique, puis émettait les 7 jours d'un coup. Le modèle ne
    voyait donc aucune dynamique. Cette version conserve l'axe temporel de bout
    en bout (T_in = T_out = 7) et émet une carte par jour de prévision."""
    def __init__(self, cfg):
        super().__init__()
        assert cfg.input_window == cfg.output_window, \
            'Cette tête 3D suppose T_in == T_out'
        self.n_out = cfg.n_out
        self.conv3d = nn.Sequential(
            nn.Conv3d(cfg.n_in, 32, (3,3,3), padding=(1,1,1)),
            nn.BatchNorm3d(32), nn.ReLU(inplace=True),
            nn.Conv3d(32, 64, (3,3,3), padding=(1,1,1)),
            nn.BatchNorm3d(64), nn.ReLU(inplace=True),
            nn.Conv3d(64, 128, (3,3,3), padding=(1,1,1)),
            nn.BatchNorm3d(128), nn.ReLU(inplace=True),
        )
        self.head = nn.Conv3d(128, cfg.n_out, 1)   # 1×1×1 : projection par jour

    def forward(self, x, H=None, W=None, y=None, tfr=0.0):
        B, T, C, H_in, W_in = x.size()
        x = x.permute(0, 2, 1, 3, 4)                # (B,C,T,H,W)
        x = self.conv3d(x)                          # (B,128,T,H,W) — T préservé
        x = self.head(x)                            # (B,n_out,T,H,W)
        return x.permute(0, 2, 1, 3, 4)             # (B,T,n_out,H,W)


### 3d. PredRNN

In [ ]:
class STLSTMCell(nn.Module):
    def __init__(self, in_ch, hid_ch, ks=3):
        super().__init__()
        self.hid_ch = hid_ch
        self.conv = nn.Conv2d(in_ch + hid_ch + hid_ch, 7 * hid_ch, ks, padding=ks // 2)

    def forward(self, x, h, c, m):
        combined = torch.cat([x, h, m], dim=1)
        gates = self.conv(combined)
        i, f, g, o, i_m, f_m, g_m = torch.chunk(gates, 7, dim=1)
        i, f, o = torch.sigmoid(i), torch.sigmoid(f), torch.sigmoid(o)
        g = torch.tanh(g)
        i_m, f_m = torch.sigmoid(i_m), torch.sigmoid(f_m)
        g_m = torch.tanh(g_m)
        c_next = f * c + i * g
        m_next = f_m * m + i_m * g_m
        h_next = o * torch.tanh(c_next + m_next)
        return h_next, c_next, m_next


class PredRNN(nn.Module):
    """PredRNN simplifié (2 couches ST-LSTM).
    FIX: hidden_dim=64 par défaut (était 32 → comparaison inéquitable avec le
    ConvLSTM à 64 filtres) + décodeur auto-régressif dans l'espace des cibles
    via une couche d'embedding 1×1, ce qui rend le teacher forcing possible."""
    def __init__(self, cfg, hidden_dim=None):
        super().__init__()
        hd = hidden_dim or cfg.predrnn_hidden
        self.n_out   = cfg.n_out
        self.out_win = cfg.output_window
        self.hidden_dim = hd
        self.cell1 = STLSTMCell(cfg.n_in, hd)
        self.cell2 = STLSTMCell(hd, hd)
        self.embed    = nn.Conv2d(cfg.n_out, cfg.n_in, 1)   # cibles → espace d'entrée
        self.head_out = nn.Conv2d(hd, cfg.n_out, 1)

    def forward(self, x, H=None, W=None, y=None, tfr=0.0):
        B, T, C, H_in, W_in = x.size()
        h1 = torch.zeros(B, self.hidden_dim, H_in, W_in, device=x.device)
        c1 = torch.zeros_like(h1); h2 = torch.zeros_like(h1)
        c2 = torch.zeros_like(h1); m  = torch.zeros_like(h1)

        for t in range(T):
            h1, c1, m = self.cell1(x[:, t], h1, c1, m)
            h2, c2, m = self.cell2(h1, h2, c2, m)

        outputs = []
        dec_raw = torch.zeros(B, self.n_out, H_in, W_in, device=x.device)
        for t in range(self.out_win):
            din = self.embed(dec_raw)
            h1, c1, m = self.cell1(din, h1, c1, m)
            h2, c2, m = self.cell2(h1, h2, c2, m)
            out_t = self.head_out(h2)
            outputs.append(out_t.unsqueeze(1))
            if (y is not None) and (torch.rand(1).item() < tfr):
                dec_raw = y[:, t]
            else:
                dec_raw = out_t
        return torch.cat(outputs, dim=1)


# Test de forme avec un batch factice

In [ ]:
# Test de forme avec un batch factice + budget de paramètres par modèle
dummy = torch.randn(2, cfg.input_window, cfg.n_in, H, W).to(DEVICE)

def n_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

cl = Seq2SeqConvLSTM(cfg).to(DEVICE)
cn = CNN3D(cfg).to(DEVICE)
pr = PredRNN(cfg).to(DEVICE)

for name, m in [('ConvLSTM', cl), ('3D-CNN', cn), ('PredRNN', pr)]:
    out = m(dummy, H, W)
    print(f'{name:<10} → sortie {tuple(out.shape)}  |  {n_params(m)/1e6:.2f} M paramètres')

del cl, cn, pr, dummy
torch.cuda.empty_cache() if torch.cuda.is_available() else None


In [ ]:
def train_model(model, X_train, Y_train, X_val, Y_val, cfg, name, H, W, resume=True):
    def _to_ncthw(arr, n_ch):
        if arr.ndim != 5:
            raise ValueError(f'Expected 5D array, got {arr.ndim}D')
        if arr.shape[2] == n_ch:
            return arr
        if arr.shape[-1] == n_ch:
            return arr.transpose(0, 1, 4, 2, 3)
        raise ValueError(f'Unexpected shape {arr.shape} for n_ch={n_ch}')

    device_type = 'cuda' if DEVICE == 'cuda' else 'cpu'

    model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.learning_rate)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=8, min_lr=1e-6)
    loss_fn = nn.MSELoss()

    use_amp = (DEVICE == 'cuda') and cfg.use_amp
    scaler = torch.amp.GradScaler(device_type, enabled=use_amp)

    ckpt_dir = OUTPUT_DIR / 'checkpoints'
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    ckpt_latest = ckpt_dir / f'{name.lower()}_latest.pt'
    ckpt_best = ckpt_dir / f'{name.lower()}_best.pt'

    X_tr_cpu = torch.from_numpy(_to_ncthw(X_train, cfg.n_in)).float()
    Y_tr_cpu = torch.from_numpy(_to_ncthw(Y_train, cfg.n_out)).float()
    X_v_cpu = torch.from_numpy(_to_ncthw(X_val, cfg.n_in)).float()
    Y_v_cpu = torch.from_numpy(_to_ncthw(Y_val, cfg.n_out)).float()

    pin_memory = DEVICE == 'cuda'
    train_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(X_tr_cpu, Y_tr_cpu),
        batch_size=cfg.batch_size,
        shuffle=True,
        drop_last=False,
        pin_memory=pin_memory,
    )
    val_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(X_v_cpu, Y_v_cpu),
        batch_size=cfg.batch_size,
        shuffle=False,
        drop_last=False,
        pin_memory=pin_memory,
    )

    best_val = float('inf')
    patience = 0
    start_epoch = 1
    N = len(train_loader.dataset)
    val_count = len(val_loader.dataset)
    history = {'train_loss': [], 'val_loss': []}

    def _checkpoint_mismatches(state_dict):
        model_state = model.state_dict()
        mismatches = []
        for key, value in state_dict.items():
            if key in model_state and value.shape != model_state[key].shape:
                mismatches.append((key, tuple(value.shape), tuple(model_state[key].shape)))
        return mismatches

    if resume and ckpt_latest.exists():
        ckpt = torch.load(ckpt_latest, map_location=DEVICE)
        mismatches = _checkpoint_mismatches(ckpt['model'])
        if mismatches:
            first = mismatches[0]
            print('Checkpoint incompatible with current model; skipping resume.')
            print(f'  Example mismatch: {first[0]} ckpt{first[1]} vs model{first[2]}')
        else:
            model.load_state_dict(ckpt['model'])
            optimizer.load_state_dict(ckpt['optimizer'])
            scheduler.load_state_dict(ckpt['scheduler'])
            if use_amp and ckpt.get('scaler') is not None:
                scaler.load_state_dict(ckpt['scaler'])
            best_val = ckpt.get('best_val', best_val)
            patience = ckpt.get('patience', patience)
            history = ckpt.get('history', history)
            start_epoch = ckpt.get('epoch', 0) + 1
            print(f'Resume {name} from epoch {start_epoch - 1}')

    print(f'\n{"="*50}')
    print(f'Entraînement {name} ({cfg.epochs} epochs)')
    print(f'{"="*50}')

    for epoch in range(start_epoch, cfg.epochs + 1):
        model.train()
        total_loss = 0.0
        # Scheduled sampling : ratio de teacher forcing décroissant linéairement
        tfr = max(0.0, cfg.tf_ratio_start * (1.0 - (epoch - 1) / max(1, int(0.75 * cfg.epochs))))

        for xb, yb in train_loader:
            xb = xb.to(DEVICE, non_blocking=pin_memory)
            yb = yb.to(DEVICE, non_blocking=pin_memory)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=device_type, enabled=use_amp):
                pred = model(xb, H, W, y=yb, tfr=tfr)
                loss = loss_fn(pred, yb)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item() * xb.size(0)

        train_loss = total_loss / N if N else 0.0

        model.eval()
        val_loss_accum = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(DEVICE, non_blocking=pin_memory)
                yb = yb.to(DEVICE, non_blocking=pin_memory)
                with torch.amp.autocast(device_type=device_type, enabled=use_amp):
                    pred = model(xb, H, W)
                    batch_loss = loss_fn(pred, yb)
                val_loss_accum += batch_loss.item() * xb.size(0)
        val_loss = val_loss_accum / val_count if val_count else float('inf')

        scheduler.step(val_loss)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)

        improved = val_loss < best_val
        if improved:
            best_val = val_loss
            patience = 0
        else:
            patience += 1

        ckpt_payload = {
            'epoch': epoch,
            'model': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
            'scaler': scaler.state_dict() if use_amp else None,
            'best_val': best_val,
            'patience': patience,
            'history': history,
        }
        torch.save(ckpt_payload, ckpt_latest)
        if improved:
            torch.save(ckpt_payload, ckpt_best)

        print(f'  Epoch {epoch:3d} | train: {train_loss:.5f} | val: {val_loss:.5f} | tfr: {tfr:.2f}')

        if patience >= 20:
            print(f'Early stopping epoch {epoch}')
            break

    best_path = ckpt_best if ckpt_best.exists() else ckpt_latest
    if best_path.exists():
        best_ckpt = torch.load(best_path, map_location=DEVICE)
        model.load_state_dict(best_ckpt['model'])
    print(f'  ✅ Best val_loss: {best_val:.5f}')
    return model, history

In [ ]:
def _to_ncthw(arr, n_ch):
    if arr.ndim != 5:
        raise ValueError(f'Expected 5D array, got {arr.ndim}D')
    if arr.shape[2] == n_ch:
        return arr
    if arr.shape[-1] == n_ch:
        return arr.transpose(0, 1, 4, 2, 3)
    raise ValueError(f'Unexpected shape {arr.shape} for n_ch={n_ch}')


def predict_phys(model, X_test, norm_stats, cfg, H, W):
    """Prédictions du modèle en UNITÉS PHYSIQUES, forme (N, T_out, n_out, H, W)."""
    model.eval()
    X_te = torch.from_numpy(_to_ncthw(X_test, cfg.n_in)).float().to(DEVICE)
    preds = []
    for i in range(0, X_test.shape[0], cfg.batch_size):
        with torch.no_grad():
            preds.append(model(X_te[i:i+cfg.batch_size], H, W).cpu().numpy())
    pred = np.concatenate(preds, axis=0)
    std  = norm_stats['std'][:cfg.n_out].reshape(1, 1, cfg.n_out, 1, 1)
    mean = norm_stats['mean'][:cfg.n_out].reshape(1, 1, cfg.n_out, 1, 1)
    return pred * std + mean


def y_phys(Y_test, norm_stats, cfg):
    """Vérité terrain en unités physiques, forme (N, T_out, H, W, n_out)."""
    std  = norm_stats['std'][:cfg.n_out].reshape(1, 1, 1, 1, cfg.n_out)
    mean = norm_stats['mean'][:cfg.n_out].reshape(1, 1, 1, 1, cfg.n_out)
    return Y_test * std + mean


def metrics_from_phys(pred_phys, ytrue_phys, cfg):
    """RMSE / MAE / R2 (T_out, n_out) à partir de champs physiques."""
    rmse = np.zeros((cfg.output_window, cfg.n_out))
    mae  = np.zeros((cfg.output_window, cfg.n_out))
    r2   = np.zeros((cfg.output_window, cfg.n_out))
    for lead in range(cfg.output_window):
        for v in range(cfg.n_out):
            yt = ytrue_phys[:, lead, :, :, v]
            yp = pred_phys[:, lead, v, :, :]
            mae[lead, v]  = float(np.mean(np.abs(yt - yp)))
            rmse[lead, v] = float(np.sqrt(np.mean((yt - yp) ** 2)))
            ss_res = np.sum((yt - yp) ** 2)
            ss_tot = np.sum((yt - yt.mean()) ** 2)
            r2[lead, v] = float(1.0 - ss_res / ss_tot) if ss_tot > 0 else 0.0
    return rmse, mae, r2


def evaluate_model(model, X_test, Y_test, norm_stats, cfg, H, W):
    pred = predict_phys(model, X_test, norm_stats, cfg, H, W)
    return metrics_from_phys(pred, y_phys(Y_test, norm_stats, cfg), cfg)


## 5. Entraînement des modèles PyTorch

In [ ]:
# ── ConvLSTM ──
cl_model = Seq2SeqConvLSTM(cfg)
cl_model, cl_history = train_model(cl_model, X_train, Y_train, X_val, Y_val, cfg, 'ConvLSTM', H, W)
rmse_cl, mae_cl, r2_cl = evaluate_model(cl_model, X_test, Y_test, norm_stats, cfg, H, W)
print('ConvLSTM — RMSE par horizon (°C) :')
import pandas as pd
display(pd.DataFrame(np.round(rmse_cl, 3), columns=cfg.targets, index=[f'J+{i+1}' for i in range(cfg.output_window)]))

In [ ]:
# ── 3D‑CNN ──
cn_model = CNN3D(cfg)
cn_model, cn_history = train_model(cn_model, X_train, Y_train, X_val, Y_val, cfg, 'CNN3D', H, W)
rmse_cn, mae_cn, r2_cn = evaluate_model(cn_model, X_test, Y_test, norm_stats, cfg, H, W)
print('3D‑CNN — RMSE par horizon (°C) :')
import pandas as pd
display(pd.DataFrame(np.round(rmse_cn, 3), columns=cfg.targets, index=[f'J+{i+1}' for i in range(cfg.output_window)]))

In [ ]:
# ── PredRNN ──  (FIX: capacité alignée sur le ConvLSTM via cfg.predrnn_hidden)
pr_model = PredRNN(cfg)
pr_model, pr_history = train_model(pr_model, X_train, Y_train, X_val, Y_val, cfg, 'PredRNN', H, W)
rmse_pr, mae_pr, r2_pr = evaluate_model(pr_model, X_test, Y_test, norm_stats, cfg, H, W)
print('PredRNN — RMSE par horizon (°C) :')
import pandas as pd
display(pd.DataFrame(np.round(rmse_pr, 3), columns=cfg.targets, index=[f'J+{i+1}' for i in range(cfg.output_window)]))

In [ ]:
# ── Persistence (baseline) ──
def persistence_metrics(X_test, Y_test, norm_stats, cfg):
    last_day = X_test[:, -1, :, :, :cfg.n_out]                       # (N,H,W,n_out)
    pred = np.repeat(last_day[:, None, ...], cfg.output_window, axis=1)
    pred_ncthw = pred.transpose(0, 1, 4, 2, 3)                       # (N,T,n_out,H,W)
    std  = norm_stats['std'][:cfg.n_out].reshape(1, 1, cfg.n_out, 1, 1)
    mean = norm_stats['mean'][:cfg.n_out].reshape(1, 1, cfg.n_out, 1, 1)
    return metrics_from_phys(pred_ncthw * std + mean, y_phys(Y_test, norm_stats, cfg), cfg)

rmse_ps, mae_ps, r2_ps = persistence_metrics(X_test, Y_test, norm_stats, cfg)
print('Persistance — RMSE par horizon (°C) :')
import pandas as pd
display(pd.DataFrame(np.round(rmse_ps, 3), columns=cfg.targets, index=[f'J+{i+1}' for i in range(cfg.output_window)]))

## 6. Évaluation discriminante — au-delà de la RMSE agrégée

La RMSE moyennée sur la grille sature au plancher de prédictibilité et ne départage
pas des architectures entraînées avec la même MSE (elles approximent toutes la même
espérance conditionnelle $E[Y|X]$). Les cellules suivantes ajoutent les métriques
qui discriminent réellement les modèles pour un système d'alerte :

1. **Baseline climatologique** (moyenne par jour de l'année, fenêtre ±15 j) — la
   vraie référence à moyen terme, plus exigeante que la persistance à J+5/J+7 ;
2. **Seuils percentiles P90 par pixel/jour calendaire** (train uniquement) — la
   même définition que la chaîne d'alerte du mémoire ;
3. **RMSE conditionnée aux extrêmes** (jours où l'observation dépasse son P90
   local) — là où le *blurring* pénalise les modèles lisses ;
4. **POD / FAR / CSI** du déclencheur journalier conjoint (Tmax>P90 **et**
   Tmin>P90) par échéance — la qualité de la *décision*, pas du champ ;
5. **Bootstrap par blocs** de la différence de RMSE entre modèles — pour affirmer
   (ou non) qu'un écart est statistiquement significatif.

In [ ]:
# ── Climatologie journalière & seuils percentiles (TRAIN uniquement, ±15 jours) ──
import pandas as pd

def day_of_year(times):
    doy = pd.DatetimeIndex(times).dayofyear.values
    return np.where(doy == 366, 365, doy)   # fusion 29 février

doy_train = day_of_year(times_train)

def doy_stat(raw_phys, doy, fn, window=15):
    """Statistique par jour calendaire avec fenêtre circulaire ±window jours.
    raw_phys : (T, H, W) — UNE variable en unités physiques.
    Retourne (366, H, W) (index 1..365 utilisés)."""
    out = np.full((366,) + raw_phys.shape[1:], np.nan, dtype=np.float32)
    for d in range(1, 366):
        dist = np.minimum(np.abs(doy - d), 365 - np.abs(doy - d))
        mask = dist <= window
        masked_data = raw_phys[mask]
        if masked_data.shape[0] > 0:
            out[d] = fn(masked_data, axis=0)

    # Remplacer les NaN (dus à l'échantillon réduit) par la moyenne globale de la variable
    out = np.nan_to_num(out, nan=np.nanmean(out))
    return out

# Variables physiques du train (raw_train est déjà en unités physiques)
# Update channel names to match configuration
i_tmax = list(cfg.channels).index('Tmax')
i_tmin = list(cfg.channels).index('Tmin')

clim_tmax = doy_stat(raw_train[..., i_tmax], doy_train, np.mean)
clim_tmin = doy_stat(raw_train[..., i_tmin], doy_train, np.mean)
p90_tmax  = doy_stat(raw_train[..., i_tmax], doy_train, lambda a, axis: np.quantile(a, 0.90, axis=axis))
p90_tmin  = doy_stat(raw_train[..., i_tmin], doy_train, lambda a, axis: np.quantile(a, 0.90, axis=axis))

print('Climatologies et seuils P90 calculés :', clim_tmax.shape)

# Dates des cibles Y du test : fenêtre i → jours [i+input_window, i+WIN-1]
WIN = cfg.input_window + cfg.output_window
N_test_windows = X_test.shape[0]
dates_Y_test = np.stack([times_test[i + cfg.input_window : i + WIN]
                         for i in range(N_test_windows)], axis=0)    # (N, T_out)
doy_Y_test = day_of_year(dates_Y_test.reshape(-1)).reshape(N_test_windows, cfg.output_window)


In [ ]:
# ── Baseline climatologique (prédire la normale du jour) ──
def climatology_metrics(Y_test, doy_Y, cfg, norm_stats):
    yt = y_phys(Y_test, norm_stats, cfg)                  # (N,T,H,W,n_out)

    # Calculer la climatologie pour toutes les variables cibles dynamiquement
    clim_dict = {}
    for v in range(cfg.n_out):
        ch_idx = list(cfg.channels).index(cfg.targets[v])
        clim_dict[v] = doy_stat(raw_train[..., ch_idx], doy_train, np.mean)

    rmse = np.zeros((cfg.output_window, cfg.n_out))
    mae  = np.zeros((cfg.output_window, cfg.n_out))
    r2   = np.zeros((cfg.output_window, cfg.n_out))
    for lead in range(cfg.output_window):
        d = doy_Y[:, lead]                                # (N,)
        for v in range(cfg.n_out):
            yp = clim_dict[v][d]                               # (N,H,W)
            ytl = yt[:, lead, :, :, v]
            mae[lead, v]  = float(np.mean(np.abs(ytl - yp)))
            rmse[lead, v] = float(np.sqrt(np.mean((ytl - yp) ** 2)))
            ss_res = np.sum((ytl - yp) ** 2)
            ss_tot = np.sum((ytl - ytl.mean()) ** 2)
            r2[lead, v] = float(1 - ss_res / ss_tot) if ss_tot > 0 else 0.0
    return rmse, mae, r2

rmse_cm, mae_cm, r2_cm = climatology_metrics(Y_test, doy_Y_test, cfg, norm_stats)
print('Climatologie — RMSE par horizon (°C) :')
import pandas as pd
display(pd.DataFrame(np.round(rmse_cm, 3), columns=cfg.targets, index=[f'J+{i+1}' for i in range(cfg.output_window)]))

In [ ]:
# ── RMSE conditionnée aux EXTRÊMES (obs > P90 local du jour) ──
def extreme_rmse(pred_phys, Y_test, doy_Y, cfg, norm_stats):
    """RMSE calculée uniquement sur les pixels-jours où l'OBSERVATION dépasse
    son P90 climatologique local — la zone qui compte pour l'alerte canicule."""
    yt = y_phys(Y_test, norm_stats, cfg)
    p90 = {0: p90_tmax, 1: p90_tmin}
    out = np.full((cfg.output_window, cfg.n_out), np.nan)
    frac = np.zeros_like(out)
    for lead in range(cfg.output_window):
        d = doy_Y[:, lead]
        for v in p90.keys():
            thr = p90[v][d]                                # (N,H,W)
            ytl = yt[:, lead, :, :, v]
            ypl = pred_phys[:, lead, v, :, :]
            mask = ytl > thr
            frac[lead, v] = mask.mean()
            if mask.any():
                out[lead, v] = float(np.sqrt(np.mean((ytl[mask] - ypl[mask]) ** 2)))
    return out, frac

# Prédictions physiques de chaque modèle (réutilisées partout ensuite)
pred_cl = predict_phys(cl_model, X_test, norm_stats, cfg, H, W)
pred_cn = predict_phys(cn_model, X_test, norm_stats, cfg, H, W)
pred_pr = predict_phys(pr_model, X_test, norm_stats, cfg, H, W)

ext = {}
for name, p in [('ConvLSTM', pred_cl), ('3D-CNN', pred_cn), ('PredRNN', pred_pr)]:
    ext[name], frac_ext = extreme_rmse(p, Y_test, doy_Y_test, cfg, norm_stats)
    print(f'{name:<10} RMSE-extrêmes Tmax J+1→J+7 : {np.round(ext[name][:,0],2)}')
print(f'(fraction de pixels-jours extrêmes ≈ {frac_ext[:, 0].mean()*100:.1f} %)')

In [ ]:
# ── POD / FAR / CSI du déclencheur conjoint journalier (Tmax>P90 ET Tmin>P90) ──
def event_scores(pred_phys, Y_test, doy_Y, cfg, norm_stats):
    yt = y_phys(Y_test, norm_stats, cfg)
    pod = np.zeros(cfg.output_window); far = np.zeros(cfg.output_window)
    csi = np.zeros(cfg.output_window)
    for lead in range(cfg.output_window):
        d = doy_Y[:, lead]
        thrx, thrn = p90_tmax[d], p90_tmin[d]
        obs  = (yt[:, lead, :, :, 0] > thrx) & (yt[:, lead, :, :, 1] > thrn)
        prd  = (pred_phys[:, lead, 0] > thrx) & (pred_phys[:, lead, 1] > thrn)
        hits   = np.sum(obs & prd)
        misses = np.sum(obs & ~prd)
        fa     = np.sum(~obs & prd)
        pod[lead] = hits / (hits + misses) if (hits + misses) else np.nan
        far[lead] = fa / (hits + fa) if (hits + fa) else np.nan
        csi[lead] = hits / (hits + misses + fa) if (hits + misses + fa) else np.nan
    return pod, far, csi

print(f"{'Modèle':<10} | {'POD J+1/J+3/J+7':<22} | {'FAR J+1/J+3/J+7':<22} | {'CSI J+1/J+3/J+7'}")
event_results = {}
for name, p in [('ConvLSTM', pred_cl), ('3D-CNN', pred_cn), ('PredRNN', pred_pr)]:
    pod, far, csi = event_scores(p, Y_test, doy_Y_test, cfg, norm_stats)
    event_results[name] = (pod, far, csi)
    fmt = lambda a: '/'.join(f'{a[i]:.2f}' for i in [0, 2, 6])
    print(f'{name:<10} | {fmt(pod):<22} | {fmt(far):<22} | {fmt(csi)}')


In [ ]:
# ── Significativité : bootstrap par blocs de la différence de RMSE ──
def block_bootstrap_diff(predA, predB, Y_test, cfg, norm_stats, lead, v=0,
                         n_boot=1000, block=14, seed=0):
    """IC à 95 % de RMSE(A) − RMSE(B) au jour `lead` pour la variable v.
    Bootstrap par blocs temporels (les fenêtres successives sont corrélées)."""
    yt = y_phys(Y_test, norm_stats, cfg)[:, lead, :, :, v]
    eA = (predA[:, lead, v] - yt) ** 2     # (N,H,W) erreurs quadratiques
    eB = (predB[:, lead, v] - yt) ** 2
    sA = eA.mean(axis=(1, 2)); sB = eB.mean(axis=(1, 2))   # MSE spatiale par échantillon
    N = len(sA); rng = np.random.default_rng(seed)
    diffs = np.empty(n_boot)
    n_blocks = int(np.ceil(N / block))
    for b in range(n_boot):
        starts = rng.integers(0, max(1, N - block), size=n_blocks)
        idx = np.concatenate([np.arange(s, s + block) for s in starts])[:N]
        diffs[b] = np.sqrt(sA[idx].mean()) - np.sqrt(sB[idx].mean())
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return float(np.sqrt(sA.mean()) - np.sqrt(sB.mean())), (float(lo), float(hi))

for lead in [0, 2, 6]:
    d, (lo, hi) = block_bootstrap_diff(pred_cl, pred_pr, Y_test, cfg, norm_stats, lead)
    signif = 'SIGNIFICATIF' if (lo > 0 or hi < 0) else 'non significatif'
    print(f'J+{lead+1}  ΔRMSE(ConvLSTM − PredRNN) Tmax = {d:+.3f} °C  IC95 [{lo:+.3f}, {hi:+.3f}]  → {signif}')


In [ ]:
# ── Courbes de perte ──
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))

ax1.plot(cl_history['train_loss'], label='Train', color='#1f77b4')
ax1.plot(cl_history['val_loss'],   label='Val',   color='#ff7f0e')
ax1.set_title('ConvLSTM — Loss', fontsize=13, fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('MSE')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(cn_history['train_loss'], label='Train', color='#1f77b4')
ax2.plot(cn_history['val_loss'],   label='Val',   color='#ff7f0e')
ax2.set_title('3D‑CNN — Loss', fontsize=13, fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('MSE')
ax2.legend(); ax2.grid(True, alpha=0.3)

ax3.plot(pr_history['train_loss'], label='Train', color='#1f77b4')
ax3.plot(pr_history['val_loss'],   label='Val',   color='#ff7f0e')
ax3.set_title('PredRNN — Loss', fontsize=13, fontweight='bold')
ax3.set_xlabel('Epoch'); ax3.set_ylabel('MSE')
ax3.legend(); ax3.grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'training_loss.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Graphiques par horizon (RMSE/MAE/R2)

In [ ]:
results = {
    'ConvLSTM':    (rmse_cl,  mae_cl,  r2_cl),
    'Climatologie': (rmse_cm, mae_cm, r2_cm),
    '3D-CNN':      (rmse_cn,  mae_cn,  r2_cn),
    'PredRNN':     (rmse_pr,  mae_pr,  r2_pr),
    'Persistence': (rmse_ps,  mae_ps,  r2_ps),
}

target_names = ['$T_{max}$', '$T_{min}$', 'Humidité relative', 'Vent zonal (u10)', 'Vent méridien (v10)']
target_vars  = ['tmax', 'tmin', 'rh', 'u10', 'v10']
units = ['°C', '°C', '%', 'm/s', 'm/s']

colors  = {
    'ConvLSTM': '#1f77b4',
    'Climatologie': '#9467bd',
    '3D-CNN': '#ff7f0e',
    'PredRNN': '#2ca02c',
    'Persistence': '#7f7f7f',
}
markers = {
    'ConvLSTM': 'o',
    'Climatologie': 'd',
    '3D-CNN': 's',
    'PredRNN': '^',
    'Persistence': 'x',
}

leads = np.arange(1, 8)

for metric_name in ['RMSE', 'MAE', 'R2']:
    for v_idx in range(cfg.n_out):
        fig, ax = plt.subplots(figsize=(7, 5))

        for mdl_name, (rm, ma, r2) in results.items():
            if metric_name == 'RMSE':
                vals = rm[:, v_idx]
            elif metric_name == 'MAE':
                vals = ma[:, v_idx]
            else:
                vals = r2[:, v_idx]
            ax.plot(leads, vals,
                    color=colors[mdl_name], marker=markers[mdl_name],
                    markersize=7, linewidth=2, label=mdl_name)

        unit = units[v_idx] if metric_name in ('RMSE', 'MAE') else ''
        ylabel = f'{metric_name} ({unit})' if unit else metric_name
        ax.set_xlabel('Jour de prévision', fontsize=12)
        ax.set_ylabel(ylabel, fontsize=12)
        ax.set_title(f'{target_names[v_idx]} — {metric_name} par horizon',
                     fontsize=13, fontweight='bold')
        ax.set_xticks(leads)
        ax.legend(frameon=True, fontsize=10)
        ax.grid(True, alpha=0.3)
        ax.set_xlim(0.5, 7.5)
        fig.tight_layout()

        fname = OUTPUT_DIR / f'lead_time_{metric_name.lower()}_{target_vars[v_idx]}.png'
        fig.savefig(fname, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'Saved: {fname}')


## 9. Tableau récapitulatif

In [ ]:
print('\n' + '=' * 120)
print('RÉSUMÉ — RMSE / MAE / R2 moyens (J+1 → J+7)')
print('=' * 120)
header = f"{'Modèle':<14}"
for t in target_vars:
    header += f' | {t.upper():>8} RMSE | {t.upper():>8} MAE | {t.upper():>8} R2'
print(header)
print('-' * 120)

for model_name, (rm, ma, r2) in results.items():
    line = f'{model_name:<14}'
    for v in range(cfg.n_out):
        line += f' |  {rm[:, v].mean():6.2f} {units[v]}  |  {ma[:, v].mean():6.2f} {units[v]}  |  {r2[:, v].mean():6.3f}'
    print(line)
print('=' * 120)

## 10. Sauvegarde des artefacts

In [ ]:
# ── Stats de normalisation (pour le déploiement) ──
np.savez_compressed(
    OUTPUT_DIR / 'normalization.npz',
    mean=norm_stats['mean'], std=norm_stats['std'],
    channels=np.array(cfg.channels),
    target_channels=np.array(cfg.targets),
)

# ── CSV des métriques par horizon ──
with open(OUTPUT_DIR / 'horizon_metrics.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['model', 'horizon', 'variable', 'rmse', 'mae', 'r2'])
    for mdl, (rm, ma, r2) in results.items():
        for h in range(cfg.output_window):
            for v in range(cfg.n_out):
                w.writerow([mdl, f'J+{h+1}', cfg.targets[v],
                            round(rm[h, v], 3), round(ma[h, v], 3), round(r2[h, v], 4)])

# ── Config JSON ──
with open(OUTPUT_DIR / 'config.json', 'w') as f:
    config_dict = {k: v for k, v in asdict(cfg).items() if not k.startswith('_')}
    config_dict['grid'] = {'lat': H, 'lon': W}
    json.dump(config_dict, f, indent=2, default=str)

print('✅ Artefacts sauvegardés dans :', cfg.output_dir)
print()
for f in sorted(OUTPUT_DIR.glob('*')):
    print(f'  {f.name}  ({f.stat().st_size / 1024:.0f} KB)')


---

### Fichiers produits dans `training_artifacts/`

| Fichier | Description |
|---|---|
| `convlstm_best.pth` | Poids du ConvLSTM (modèle principal) |
| `cnn3d_best.pth` | Poids du 3D‑CNN |
| `predrnn_best.pth` | Poids du PredRNN |
| `normalization.npz` | Moyenne et écart‑type par canal (Z‑score) |
| `horizon_metrics.csv` | RMSE/MAE/R2 par jour de prévision et variable |
| `config.json` | Configuration de l'expérience |
| `training_loss.png` | Courbes de perte train/val |
| `lead_time_rmse_*.png` | RMSE par horizon (3 fichiers) |
| `lead_time_mae_*.png` | MAE par horizon (3 fichiers) |
| `lead_time_r2_*.png` | R2 par horizon (3 fichiers) |

**Prochaine étape** : copier `convlstm_best.pth` et `normalization.npz` dans `backend/artifacts/` pour le déploiement opérationnel.